# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`
This notebook provides a step-by-step walkthrough for loading and exploring the [FAIR2 dataset](https://doi.org/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(metadata.name)
print('\nDescription:')
print(metadata.description)

## 2. Data Overview
Review available record sets, field IDs, and column IDs.

In [ ]:
# List all record sets in the dataset using their `@id`
record_sets = list(dataset.record_sets)
print("Available record sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")

# For each record set, show its fields and their @id
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} (name: {rs.get('name', 'N/A')})")
    print(" Fields:")
    fields = rs.get('fields', [])
    for field in fields:
        print(f"   - {field['@id']} (name: {field.get('name', 'N/A')})")
        columns = field.get('columns', [])
        if columns:
            print("     Columns:")
            for col in columns:
                print(f"       - {col['@id']} (name: {col.get('name', 'N/A')})")

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Extract data for each record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nColumns in record set {record_set_id}:")
        print(df.columns.tolist())
        print(f"First 5 rows of record set {record_set_id}:")
        display(df.head())

# For further analysis, select the primary record set (pick the largest one / most relevant as default)
if len(dataframes) > 0:
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k].columns))
    print(f"\nMain record set selected for analysis: {main_record_set_id}")
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Now let's apply common data processing steps, such as filtering records based on a numeric field, normalizing it, and computing group summaries if appropriate. All field references use their `@id`.

In [ ]:
# We'll select a numeric field for analysis.
# Get numeric columns in `main_record_set_id` dataframe
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]

    # Attempt to infer a numeric field by checking dtypes
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        # Try to convert columns that look numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except:
                continue
        numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Selected numeric field for EDA: {numeric_field_id}")
        # Filter records with value above threshold
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records from {main_record_set_id} with {numeric_field_id} > {threshold} : {len(filtered_df)} records")

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by a categorical field
        group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"\nGrouping by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA in the selected record set.")
else:
    print("No record sets loaded for analysis.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and the group means if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If we grouped, plot a barplot
    if 'group_field_id' in locals():
        plt.figure(figsize=(10,5))
        grouped_df.plot(kind='bar')
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook provided a guided exploration of the FAIR2 Clinical Oncology dataset using the `mlcroissant` library. We loaded schema and metadata, investigated available record sets and their fields (referenced always by `@id`), loaded structured records to DataFrames, applied data processing steps such as filtering and normalization, and visualized selected features.

- Data from each record set were loaded dynamically by reference.
- Numeric fields were automatically identified and used for statistical analysis and visualization.
- Record sets, fields, and columns were always referenced by their schema `@id`.

Feel free to further analyze and visualize the dataset as needed for your research or data science tasks.